# Câu 1 - Thuật toán UCS

Cho `n` cái gáo nước. Gáo thứ `i` chứa tối đa `a_i` lít nước. Cần múc đúng `M` lít nước từ bờ sông qua bể nước lớn với số thao tác phù hợp theo thuật toán UCS, không được múc quá và cũng không được múc thiếu.

Các thao tác hợp lệ:

- Múc nước từ bờ sông vào một gáo cho đến khi gáo đầy.
- Chuyển nước từ một gáo sang gáo khác cho đến khi gáo nguồn hết nước hoặc gáo đích đầy.
- Chuyển toàn bộ nước trong một gáo sang bể nếu không làm bể vượt quá `M` lít.
- Vứt bỏ toàn bộ nước trong một gáo. Thao tác vứt bỏ cũng được tính là `1` thao tác.

Dữ liệu mặc định:

```text
3 13 7 8 9
```

Ý nghĩa: `n = 3`, `M = 13`, ba gáo có dung tích lần lượt là `7`, `8`, `9` lít.

Dán code:

In [ ]:
from pathlib import Path
import heapq
import math
import sys


DEFAULT_INPUT = "3 13 7 8 9"


def parse_input(input_text):
    input_text = input_text.replace("\ufeff", "").strip()
    values = [int(value) for value in input_text.split()]
    if len(values) < 3:
        raise ValueError("Can nhap n, M va danh sach dung tich cac gao")
    n = values[0]
    target = values[1]
    capacities = values[2:]
    if len(capacities) != n:
        raise ValueError("So luong dung tich gao khong khop voi n")
    return n, target, capacities


def normalize_input_text(input_text):
    input_text = input_text.replace("\ufeff", "").strip()
    if not input_text:
        input_text = DEFAULT_INPUT
    return " ".join(input_text.split())


def has_possible_answer(target, capacities):
    return target % math.gcd(*capacities) == 0


def format_state(jug_amounts, tank_amount):
    jug_text = ", ".join(
        f"Gao {index + 1}: {amount} lit"
        for index, amount in enumerate(jug_amounts)
    )
    return f"({jug_text}, Be: {tank_amount} lit)"


def generate_neighbors(state, capacities, target):
    jug_amounts, tank_amount = state
    n = len(capacities)
    neighbors = []
    for i in range(n):
        if jug_amounts[i] < capacities[i]:
            new_jugs = list(jug_amounts)
            amount = capacities[i] - jug_amounts[i]
            new_jugs[i] = capacities[i]
            new_state = (tuple(new_jugs), tank_amount)
            action = (
                f"Chuyen/Muc {amount} lit nuoc tu bo song qua gao {i + 1} "
                f"{format_state(new_state[0], new_state[1])}"
            )
            neighbors.append((new_state, action))
    for i in range(n):
        if jug_amounts[i] > 0 and tank_amount + jug_amounts[i] <= target:
            new_jugs = list(jug_amounts)
            amount = new_jugs[i]
            new_jugs[i] = 0
            new_tank = tank_amount + amount
            new_state = (tuple(new_jugs), new_tank)
            action = (
                f"Chuyen/Muc {amount} lit nuoc tu gao {i + 1} qua be "
                f"{format_state(new_state[0], new_state[1])}"
            )
            neighbors.append((new_state, action))
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            if jug_amounts[i] > 0 and jug_amounts[j] < capacities[j]:
                amount = min(jug_amounts[i], capacities[j] - jug_amounts[j])
                new_jugs = list(jug_amounts)
                new_jugs[i] -= amount
                new_jugs[j] += amount
                new_state = (tuple(new_jugs), tank_amount)
                action = (
                    f"Chuyen/Muc {amount} lit nuoc tu gao {i + 1} qua gao {j + 1} "
                    f"{format_state(new_state[0], new_state[1])}"
                )
                neighbors.append((new_state, action))
    for i in range(n):
        if jug_amounts[i] > 0:
            new_jugs = list(jug_amounts)
            amount = new_jugs[i]
            new_jugs[i] = 0
            new_state = (tuple(new_jugs), tank_amount)
            action = (
                f"Vut bo toan bo {amount} lit nuoc cua gao {i + 1}. "
                f"{format_state(new_state[0], new_state[1])}"
            )
            neighbors.append((new_state, action))
    return neighbors


def reconstruct_actions(parent, goal_state):
    actions = []
    current = goal_state
    while parent[current][0] is not None:
        previous_state, action = parent[current]
        actions.append(action)
        current = previous_state
    actions.reverse()
    return actions


def format_solution(actions):
    if actions is None:
        return "Khong co dap an"
    lines = [f"So thao tac: {len(actions)}"]
    lines.extend(f"{index}. {action}" for index, action in enumerate(actions, start=1))
    return "\n".join(lines)


def ucs_water_jug(n, target, capacities):
    if n <= 0 or target <= 0 or any(capacity <= 0 for capacity in capacities):
        return None
    if not has_possible_answer(target, capacities):
        return None

    start_state = (tuple([0] * n), 0)
    frontier = []
    order = 0
    heapq.heappush(frontier, (0, order, start_state))

    parent = {start_state: (None, None)}
    cost = {start_state: 0}
    visited = set()

    while frontier:
        current_cost, _, current_state = heapq.heappop(frontier)

        if current_state in visited:
            continue

        visited.add(current_state)
        _, tank_amount = current_state

        if tank_amount == target:
            return reconstruct_actions(parent, current_state), len(visited)

        for next_state, action in generate_neighbors(current_state, capacities, target):
            next_cost = current_cost + 1

            if next_state not in cost or next_cost < cost[next_state]:
                cost[next_state] = next_cost
                parent[next_state] = (current_state, action)
                order += 1
                heapq.heappush(frontier, (next_cost, order, next_state))

    return None


def solve(input_text):
    n, target, capacities = parse_input(input_text)
    result = ucs_water_jug(n, target, capacities)

    if result is None:
        return "Khong co dap an"

    actions, visited_count = result
    return f"So trang thai da xet: {visited_count}\n{format_solution(actions)}"


def main():
    current_dir = Path(__file__).resolve().parent
    input_text = normalize_input_text(sys.stdin.read())
    output_text = solve(input_text)
    output_file = current_dir / "UCS_out.txt"

    output_file.write_text(output_text, encoding="utf-8")
    print(f"Nhap: {input_text}")
    print(output_text)
    print(f"Da luu ket qua vao: {output_file}")


print(solve(DEFAULT_INPUT))


## Giải thích tác dụng của từng hàm

- `parse_input(input_text)`: đọc nội dung đầu vào, tách số lượng gáo `n`, mục tiêu `M` và danh sách sức chứa của các gáo.
- `normalize_input_text(input_text)`: chuẩn hóa nội dung input trước khi xử lý, giúp chương trình đọc dữ liệu ổn định hơn khi input có khoảng trắng hoặc xuống dòng khác nhau.
- `has_possible_answer(target, capacities)`: kiểm tra nhanh bài toán có khả năng có lời giải hay không dựa trên mục tiêu và sức chứa các gáo. Nếu không thể tạo đúng `M` lít thì thuật toán có thể dừng sớm.
- `format_state(jug_amounts, tank_amount)`: chuyển một trạng thái thành chuỗi dễ đọc, gồm lượng nước hiện có trong từng gáo và lượng nước trong bể.
- `generate_neighbors(state, capacities, target)`: sinh các trạng thái kế tiếp có thể đi tới từ trạng thái hiện tại bằng các thao tác hợp lệ, đồng thời kèm mô tả thao tác đã thực hiện.
- `reconstruct_actions(parent, goal_state)`: truy vết từ trạng thái đích về trạng thái ban đầu dựa trên dictionary `parent`, sau đó đảo ngược để lấy chuỗi thao tác đúng thứ tự.
- `format_solution(actions)`: định dạng danh sách thao tác lời giải thành văn bản để in ra file hoặc màn hình.
- `ucs_water_jug(n, target, capacities)`: thực hiện Uniform Cost Search cho bài toán gáo nước. Hàm dùng hàng đợi ưu tiên theo tổng chi phí từ trạng thái ban đầu, luôn mở rộng trạng thái có chi phí nhỏ nhất trước.
- `solve(input_text)`: điều phối toàn bộ quá trình: đọc input, kiểm tra điều kiện có lời giải, gọi thuật toán tương ứng và trả về nội dung kết quả.
- `main()`: hàm chạy chính khi thực thi file. Hàm đọc dữ liệu đầu vào, gọi `solve()`, in kết quả và/hoặc ghi kết quả ra file output.

## Giải thích bài toán

Mỗi trạng thái được biểu diễn bằng:

```text
((x1, x2, ..., xn), B)
```

Trong đó:

- `x_i` là lượng nước hiện có trong gáo thứ `i`.
- `B` là lượng nước đã chuyển vào bể.
- Trạng thái bắt đầu là `((0, 0, ..., 0), 0)`.
- Trạng thái đích là trạng thái có `B = M`.

Với dữ liệu `3 13 7 8 9`, trạng thái ban đầu là:

```text
((0, 0, 0), 0)
```

Từ một trạng thái hiện tại, chương trình sinh các trạng thái kế tiếp bằng bốn nhóm thao tác: múc đầy một gáo từ bờ sông, chuyển nước từ gáo vào bể, chuyển nước giữa hai gáo, hoặc vứt bỏ nước trong một gáo.

Trước khi tìm kiếm, chương trình kiểm tra điều kiện có thể có đáp án:

```text
M % gcd(a1, a2, ..., an) == 0
```

Nếu `M` không chia hết cho ước chung lớn nhất của các dung tích gáo thì không thể đo chính xác `M` lít bằng các gáo đã cho.

## Giải thích thuật toán UCS

UCS luôn mở rộng trạng thái có tổng chi phí thật nhỏ nhất từ trạng thái đầu đến trạng thái hiện tại.

Trong bài này, mỗi thao tác có chi phí bằng `1`, nên chi phí của một trạng thái chính là số thao tác đã thực hiện.

Trong chương trình:

- `frontier` là hàng đợi ưu tiên theo `current_cost`.
- `cost[state]` lưu chi phí tốt nhất đã biết để đến trạng thái đó.
- `parent` lưu trạng thái trước và hành động dùng để truy vết.
- `visited` tránh xử lý lại trạng thái đã lấy ra khỏi heap.

Công thức cập nhật chi phí:

```text
cost(next_state) = cost(current_state) + 1
```

Khi UCS lấy được trạng thái đích ra khỏi hàng đợi ưu tiên, lời giải đó là lời giải có chi phí nhỏ nhất theo cách tính chi phí của bài.

Dán kết quả thực thi:

```text
Nhap: 3 13 7 8 9
So trang thai da xet: 479
So thao tac: 7
1. Chuyen/Muc 7 lit nuoc tu bo song qua gao 1 (Gao 1: 7 lit, Gao 2: 0 lit, Gao 3: 0 lit, Be: 0 lit)
2. Chuyen/Muc 8 lit nuoc tu bo song qua gao 2 (Gao 1: 7 lit, Gao 2: 8 lit, Gao 3: 0 lit, Be: 0 lit)
3. Chuyen/Muc 7 lit nuoc tu gao 1 qua be (Gao 1: 0 lit, Gao 2: 8 lit, Gao 3: 0 lit, Be: 7 lit)
4. Chuyen/Muc 7 lit nuoc tu bo song qua gao 1 (Gao 1: 7 lit, Gao 2: 8 lit, Gao 3: 0 lit, Be: 7 lit)
5. Chuyen/Muc 7 lit nuoc tu gao 1 qua gao 3 (Gao 1: 0 lit, Gao 2: 8 lit, Gao 3: 7 lit, Be: 7 lit)
6. Chuyen/Muc 2 lit nuoc tu gao 2 qua gao 3 (Gao 1: 0 lit, Gao 2: 6 lit, Gao 3: 9 lit, Be: 7 lit)
7. Chuyen/Muc 6 lit nuoc tu gao 2 qua be (Gao 1: 0 lit, Gao 2: 0 lit, Gao 3: 9 lit, Be: 13 lit)
```

## Nhận xét

UCS tìm được lời giải gồm `7` thao tác và xét `479` trạng thái. Vì mọi thao tác đều có chi phí bằng `1`, tổng chi phí của lời giải cũng chính là số thao tác.

Điểm mạnh của UCS là cách chọn trạng thái dựa trên chi phí thật đã đi. Nếu bài toán thay đổi để mỗi loại thao tác có chi phí khác nhau, UCS vẫn có thể tìm lời giải có tổng chi phí nhỏ nhất theo trọng số thao tác đã định nghĩa.